In [4]:
## notebookutils.runtime.context

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 6, Finished, Available, Finished, False)

In [5]:
## 01 Imports

from pyspark.sql.functions import current_timestamp

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 7, Finished, Available, Finished, False)

In [6]:
## 02 Configuration

standard_sources = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

raw_path = "Files/olist_adls"

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 8, Finished, Available, Finished, False)

In [7]:
## 03 Standard CSV ingestion

for table_name, file_name in standard_sources.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/{file_name}")
        .withColumn("ingested_at", current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"bronze_{table_name}")
    )

    print(f"Loaded bronze_{table_name}: {df.count():,} rows")

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 9, Finished, Available, Finished, False)

Loaded bronze_customers: 99,441 rows
Loaded bronze_geolocation: 1,000,163 rows
Loaded bronze_order_items: 112,650 rows
Loaded bronze_order_payments: 103,886 rows
Loaded bronze_orders: 99,441 rows
Loaded bronze_products: 32,951 rows
Loaded bronze_sellers: 3,095 rows
Loaded bronze_category_translation: 71 rows


In [8]:
## 04 Review special ingestion

df_reviews = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"{raw_path}/olist_order_reviews_dataset.csv")
    .withColumn("ingested_at", current_timestamp())
)

(
    df_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_order_reviews")
)

print(f"Loaded bronze_order_reviews: {df_reviews.count():,} rows")

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 10, Finished, Available, Finished, False)

Loaded bronze_order_reviews: 99,224 rows


In [9]:
## 05 Validation

bronze_tables = [
    "bronze_customers",
    "bronze_geolocation",
    "bronze_order_items",
    "bronze_order_payments",
    "bronze_order_reviews",
    "bronze_orders",
    "bronze_products",
    "bronze_sellers",
    "bronze_category_translation"
]

for table in bronze_tables:
    df = spark.table(table)
    print(f"{table}: {df.count():,} rows | {len(df.columns)} columns")

StatementMeta(, 27b0085d-f3a6-4ace-9e5f-c17fd36969fb, 11, Finished, Available, Finished, True)

bronze_customers: 99,441 rows | 6 columns
bronze_geolocation: 1,000,163 rows | 6 columns
bronze_order_items: 112,650 rows | 8 columns
bronze_order_payments: 103,886 rows | 6 columns
bronze_order_reviews: 99,224 rows | 8 columns
bronze_orders: 99,441 rows | 9 columns
bronze_products: 32,951 rows | 10 columns
bronze_sellers: 3,095 rows | 5 columns
bronze_category_translation: 71 rows | 3 columns
